# 📧 Phishing Email Detection + Blockchain Storage
**Pipeline:**
1. Load & clean datasets (SpamAssassin + Nigerian Fraud)
2. Preprocess text (tokenize, remove stopwords, lemmatize)
3. Train & compare: Logistic Regression, Random Forest, SVM, LSTM
4. Pick the best model
5. Parse real `.eml` email files
6. Classify → store **safe email hashes** on blockchain (Ganache)
7. Verify stored emails on-chain

> ⚠️ **Security Note:** Never commit private keys to version control.
> Store them in a `.env` file and load with `python-dotenv`.

## 📦 Cell 1 — Install Dependencies

In [ ]:
# Run once to install all required packages
import subprocess, sys

packages = [
    "pandas", "numpy", "matplotlib", "seaborn",
    "scikit-learn", "nltk",
    "tensorflow",
    "web3", "py-solc-x",
    "python-dotenv",
    "protobuf<6"
]

for pkg in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

print("✅ All packages installed.")

## 📁 Cell 2 — Configuration

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

# ---------- Load secrets from .env ----------
# Create a file named ".env" next to this notebook with:
#   GANACHE_ACCOUNT=0xYourAddress
#   GANACHE_PRIVATE_KEY=0xYourPrivateKey
load_dotenv()

# ---------- Paths ----------
BASE_DIR        = Path(".")          # same folder as this notebook
SPAM_CSV        = BASE_DIR / "dataset" / "SpamAssasin.csv"
FRAUD_CSV       = BASE_DIR / "dataset" / "Nigerian_Fraud.csv"
CLEANED_CSV     = BASE_DIR / "cleaned_dataset.csv"
ARTIFACT_DIR    = BASE_DIR / "artifacts"
EMAIL_FOLDER    = BASE_DIR / "emails"
SOL_FILE        = BASE_DIR / "EmailRegistry.sol"
COMPILED_JSON   = BASE_DIR / "compiled_code.json"
DEPLOYMENT_JSON = BASE_DIR / "EmailRegistry_Deployment.json"

ARTIFACT_DIR.mkdir(exist_ok=True)

# ---------- Blockchain (Ganache) ----------
GANACHE_URL     = "http://127.0.0.1:7545"
ACCOUNT_ADDRESS = os.getenv("GANACHE_ACCOUNT",     "0xYourGanacheAddress")
PRIVATE_KEY     = os.getenv("GANACHE_PRIVATE_KEY", "0xYourPrivateKey")

# ---------- ML Params ----------
RANDOM_SEED = 42
TFIDF_FEATURES = 5000
LSTM_MAX_VOCAB = 20000
LSTM_MAX_LEN   = 150
LSTM_EMB_DIM   = 128

print("✅ Config loaded.")
print(f"   Artifacts  → {ARTIFACT_DIR.resolve()}")
print(f"   Account    → {ACCOUNT_ADDRESS[:10]}...  (loaded from .env)")

## 🧹 Cell 3 — Load & Clean Datasets

In [ ]:
import re, string
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

def load_df(path):
    """Load CSV with flexible column detection."""
    df = pd.read_csv(path)
    df.columns = [c.strip().lower() for c in df.columns]
    df.drop_duplicates(inplace=True)
    df.dropna(inplace=True)
    return df

def get_text_col(df):
    """Return name of the text column (text / message / body)."""
    for col in ["text", "message", "body"]:
        if col in df.columns:
            return col
    raise ValueError(f"No text column found. Columns: {list(df.columns)}")

def clean_text(text):
    """Lowercase, strip URLs, HTML, punctuation and digits."""
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = re.sub(r"https?://\S+|www\.\S+", "", text)   # URLs
    text = re.sub(r"<[^>]+>", "", text)                  # HTML tags
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\d+", "", text)                      # digits
    text = re.sub(r"\s+", " ", text).strip()
    return text

# ---- Load ----
spam_df  = load_df(SPAM_CSV)
fraud_df = load_df(FRAUD_CSV)

# ---- Clean text ----
spam_df["clean_text"]  = spam_df[get_text_col(spam_df)].apply(clean_text)
fraud_df["clean_text"] = fraud_df[get_text_col(fraud_df)].apply(clean_text)

# ---- Normalise labels → "phishing" or "safe" ----
# SpamAssassin uses 1/0 or 'spam'/'ham'
if spam_df["label"].dtype in ["int64", "float64"]:
    spam_df["label"] = spam_df["label"].apply(lambda x: "phishing" if x == 1 else "safe")
else:
    spam_df["label"] = spam_df["label"].str.lower().map(
        {"spam": "phishing", "ham": "safe", "phishing": "phishing", "safe": "safe"}
    )

fraud_df["label"] = "phishing"

# ---- Combine ----
combined_df = pd.concat(
    [spam_df[["clean_text", "label"]], fraud_df[["clean_text", "label"]]],
    ignore_index=True
)
combined_df.dropna(subset=["label"], inplace=True)

# ---- EDA ----
print("Dataset shape:", combined_df.shape)
print("\nClass distribution:\n", combined_df["label"].value_counts())

combined_df["text_length"] = combined_df["clean_text"].apply(len)
print("\nAvg text length by class:\n",
      combined_df.groupby("label")["text_length"].mean().round(1))

combined_df["label"].value_counts().plot(kind="bar", title="Class Distribution", color=["tomato", "steelblue"])
plt.tight_layout()
plt.show()

# ---- Top words in phishing ----
phishing_words = " ".join(combined_df[combined_df["label"] == "phishing"]["clean_text"]).split()
print("\nTop-20 phishing words:", Counter(phishing_words).most_common(20))

# ---- Save ----
combined_df.to_csv(CLEANED_CSV, index=False)
print(f"\n✅ Saved cleaned dataset → {CLEANED_CSV}")

## 🔠 Cell 4 — Text Preprocessing (Tokenize → Stopwords → Lemmatise)

In [ ]:
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

for pkg in ["punkt", "punkt_tab", "stopwords", "wordnet"]:
    nltk.download(pkg, quiet=True)

STOP_WORDS = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def preprocess_text(s: str) -> str:
    """Tokenise → lowercase → keep alpha → drop stopwords → lemmatise."""
    tokens = word_tokenize(str(s))
    tokens = [
        lemmatizer.lemmatize(w.lower())
        for w in tokens
        if w.isalpha() and w.lower() not in STOP_WORDS
    ]
    return " ".join(tokens)

df = pd.read_csv(CLEANED_CSV)
df["processed_text"] = df["clean_text"].apply(preprocess_text)

print("✅ Preprocessing complete.")
print(df[["clean_text", "processed_text", "label"]].head(3))

## ✂️ Cell 5 — TF-IDF Vectorisation + Train/Test Split

In [ ]:
import pickle
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

np.random.seed(RANDOM_SEED)

# Labels: encode to integers
labels_sorted = sorted(df["label"].unique().tolist())  # ['phishing', 'safe']
label_to_int  = {lab: i for i, lab in enumerate(labels_sorted)}
int_to_label  = {i: lab for lab, i in label_to_int.items()}

X_text = df["processed_text"].astype(str).values
y_all  = df["label"].map(label_to_int).values

X_train_txt, X_test_txt, y_train, y_test = train_test_split(
    X_text, y_all, test_size=0.2, random_state=RANDOM_SEED, stratify=y_all
)

# TF-IDF
vectorizer = TfidfVectorizer(max_features=TFIDF_FEATURES)
X_train_tfidf = vectorizer.fit_transform(X_train_txt)
X_test_tfidf  = vectorizer.transform(X_test_txt)

# Persist
with open(ARTIFACT_DIR / "tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(vectorizer, f)

import json
with open(ARTIFACT_DIR / "label_mapping.json", "w") as f:
    json.dump({"labels": labels_sorted,
               "label_to_int": label_to_int,
               "int_to_label": int_to_label}, f, indent=2)

print(f"✅ Split → train: {len(X_train_txt)}, test: {len(X_test_txt)}")
print(f"   Classes: {labels_sorted}")

## 🤖 Cell 6 — Train & Compare All Models

In [ ]:
import warnings
import seaborn as sns
import matplotlib.pyplot as plt
import tensorflow as tf

from sklearn.linear_model  import LogisticRegression
from sklearn.ensemble      import RandomForestClassifier
from sklearn.svm           import SVC
from sklearn.metrics       import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, roc_curve, auc
)
from sklearn.preprocessing import label_binarize
from tensorflow.keras.models     import Sequential
from tensorflow.keras.layers     import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.callbacks  import EarlyStopping
from tensorflow.keras.preprocessing.text     import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

warnings.filterwarnings("ignore")
tf.random.set_seed(RANDOM_SEED)

n_classes = len(labels_sorted)

# ── Shared evaluation helper ────────────────────────────────────────────────
def evaluate(y_true, y_pred, name: str) -> dict:
    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average="weighted", zero_division=0)
    rec  = recall_score   (y_true, y_pred, average="weighted", zero_division=0)
    f1   = f1_score       (y_true, y_pred, average="weighted", zero_division=0)

    print(f"\n📊 {name}")
    print(classification_report(y_true, y_pred,
                                 target_names=labels_sorted, zero_division=0))

    cm = confusion_matrix(y_true, y_pred, labels=list(range(n_classes)))
    plt.figure(figsize=(4, 3))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=labels_sorted, yticklabels=labels_sorted)
    plt.title(f"{name} – Confusion Matrix")
    plt.xlabel("Predicted"); plt.ylabel("Actual")
    plt.tight_layout()
    plt.savefig(ARTIFACT_DIR / f"cm_{name.replace(' ','_')}.png", dpi=100)
    plt.show()

    return dict(model=name, accuracy=acc, precision=prec, recall=rec, f1=f1)

def plot_roc(y_true, y_proba, name):
    """
    ROC curve that works for both binary (2 classes) and multiclass.
    
    Root cause of the IndexError:
    label_binarize with 2 classes returns shape (n, 1) not (n, 2).
    We handle this by treating binary as a special case — plot a single
    curve using the positive-class (index 1) probability column.
    """
    plt.figure(figsize=(5, 4))

    if n_classes == 2:
        # Binary: positive class is index 1; proba shape is (n, 2)
        pos_col = y_proba[:, 1] if y_proba.ndim == 2 else y_proba
        fpr, tpr, _ = roc_curve(y_true, pos_col, pos_label=1)
        roc_auc = auc(fpr, tpr)
        plt.plot(fpr, tpr,
                 label=f"{labels_sorted[1]} vs rest (AUC={roc_auc:.2f})",
                 color='darkorange', lw=2)
    else:
        # Multiclass: binarize and plot one curve per class
        y_bin = label_binarize(y_true, classes=list(range(n_classes)))
        for i, lab in enumerate(labels_sorted):
            col = y_proba[:, i] if y_proba.ndim == 2 else y_proba
            fpr, tpr, _ = roc_curve(y_bin[:, i], col)
            plt.plot(fpr, tpr, label=f"{lab} (AUC={auc(fpr, tpr):.2f})")

    plt.plot([0,1],[0,1],"k--", lw=1)
    plt.xlabel("FPR"); plt.ylabel("TPR")
    plt.title(f"{name} – ROC Curve")
    plt.legend(loc="lower right")
    plt.tight_layout()
    plt.savefig(ARTIFACT_DIR / f"roc_{name.replace(' ','_')}.png", dpi=100)
    plt.show()

results   = []          # list of metric dicts
model_zoo = {}          # name → fitted model object

# ── Logistic Regression ─────────────────────────────────────────────────────
lr = LogisticRegression(max_iter=500, n_jobs=-1, random_state=RANDOM_SEED)
lr.fit(X_train_tfidf, y_train)
lr_pred  = lr.predict(X_test_tfidf)
lr_proba = lr.predict_proba(X_test_tfidf)
res = evaluate(y_test, lr_pred, "Logistic Regression")
plot_roc(y_test, lr_proba, "Logistic Regression")
results.append(res);  model_zoo["Logistic Regression"] = lr

# ── Random Forest ────────────────────────────────────────────────────────────
rf = RandomForestClassifier(n_estimators=300, random_state=RANDOM_SEED, n_jobs=-1)
rf.fit(X_train_tfidf, y_train)
rf_pred  = rf.predict(X_test_tfidf)
rf_proba = rf.predict_proba(X_test_tfidf)
res = evaluate(y_test, rf_pred, "Random Forest")
plot_roc(y_test, rf_proba, "Random Forest")
results.append(res);  model_zoo["Random Forest"] = rf

# ── SVM ──────────────────────────────────────────────────────────────────────
svm = SVC(kernel="linear", probability=True, random_state=RANDOM_SEED)
svm.fit(X_train_tfidf, y_train)
svm_pred  = svm.predict(X_test_tfidf)
svm_proba = svm.predict_proba(X_test_tfidf)
res = evaluate(y_test, svm_pred, "SVM")
plot_roc(y_test, svm_proba, "SVM")
results.append(res);  model_zoo["SVM"] = svm

# ── LSTM ─────────────────────────────────────────────────────────────────────
tokenizer_lstm = Tokenizer(num_words=LSTM_MAX_VOCAB, oov_token="<OOV>")
tokenizer_lstm.fit_on_texts(X_train_txt)

X_tr_seq = pad_sequences(tokenizer_lstm.texts_to_sequences(X_train_txt),
                          maxlen=LSTM_MAX_LEN, padding="post", truncating="post")
X_te_seq = pad_sequences(tokenizer_lstm.texts_to_sequences(X_test_txt),
                          maxlen=LSTM_MAX_LEN, padding="post", truncating="post")

lstm = Sequential([
    Embedding(min(LSTM_MAX_VOCAB, len(tokenizer_lstm.word_index)+1),
              LSTM_EMB_DIM, input_length=LSTM_MAX_LEN),
    LSTM(128, dropout=0.3, recurrent_dropout=0.3),
    Dense(64, activation="relu"),
    Dropout(0.5),
    Dense(n_classes, activation="softmax")
])
lstm.compile(loss="sparse_categorical_crossentropy",
             optimizer="adam", metrics=["accuracy"])

lstm.fit(X_tr_seq, y_train,
         validation_data=(X_te_seq, y_test),
         epochs=8, batch_size=64,
         callbacks=[EarlyStopping(monitor="val_loss", patience=2,
                                   restore_best_weights=True)],
         verbose=1)

lstm_proba = lstm.predict(X_te_seq)
lstm_pred  = np.argmax(lstm_proba, axis=1)
res = evaluate(y_test, lstm_pred, "LSTM")
plot_roc(y_test, lstm_proba, "LSTM")
results.append(res);  model_zoo["LSTM"] = lstm

# Save LSTM tokenizer
with open(ARTIFACT_DIR / "lstm_tokenizer.json", "w") as f:
    f.write(tokenizer_lstm.to_json())

# ── Summary bar chart ────────────────────────────────────────────────────────
results_df = pd.DataFrame(results)
metrics = ["accuracy", "precision", "recall", "f1"]
x = np.arange(len(results_df))
w = 0.2
fig, ax = plt.subplots(figsize=(9, 5))
for i, m in enumerate(metrics):
    ax.bar(x + i*w, results_df[m], w, label=m.capitalize())
ax.set_xticks(x + w*1.5)
ax.set_xticklabels(results_df["model"])
ax.set_ylabel("Score")
ax.set_title("Model Comparison")
ax.legend()
plt.tight_layout()
plt.savefig(ARTIFACT_DIR / "model_comparison.png", dpi=100)
plt.show()

results_df.to_csv(ARTIFACT_DIR / "metrics.csv", index=False)
print("\n📋 Metrics:\n", results_df.to_string(index=False))

## 🏆 Cell 7 — Save Best Model

In [ ]:
best_row  = results_df.sort_values("f1", ascending=False).iloc[0]
best_name = best_row["model"]
best_obj  = model_zoo[best_name]

print(f"🥇 Best model: {best_name}  (F1 = {best_row['f1']:.4f})")

if best_name == "LSTM":
    best_obj.save(ARTIFACT_DIR / "best_model.keras")
    best_model_path = "artifacts/best_model.keras"
else:
    with open(ARTIFACT_DIR / "best_model.pkl", "wb") as f:
        pickle.dump(best_obj, f)
    best_model_path = "artifacts/best_model.pkl"

# Record which model won
with open(ARTIFACT_DIR / "best_model.txt", "w") as f:
    f.write(f"model_name: {best_name}\n")
    f.write(f"saved_as:   {best_model_path}\n")
    f.write(f"f1_score:   {best_row['f1']:.4f}\n")

print(f"✅ Saved → {best_model_path}")

## 📜 Cell 8 — Write Solidity Smart Contract

In [ ]:
solidity_code = """
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.0;

contract EmailRegistry {

    struct EmailRecord {
        string  emailHash;    // SHA-256 hash of the email body
        string  sender;       // From: field
        string  subject;      // Subject: field
        string  prediction;   // "safe" or "phishing"
        uint256 timestamp;    // Unix timestamp
    }

    EmailRecord[] private emails;

    event EmailStored(
        uint256 indexed id,
        string  emailHash,
        string  prediction,
        uint256 timestamp
    );

    /// @notice Store a new email record
    function storeEmail(
        string memory _emailHash,
        string memory _sender,
        string memory _subject,
        string memory _prediction
    ) public {
        emails.push(EmailRecord({
            emailHash : _emailHash,
            sender    : _sender,
            subject   : _subject,
            prediction: _prediction,
            timestamp : block.timestamp
        }));
        emit EmailStored(emails.length - 1, _emailHash, _prediction, block.timestamp);
    }

    /// @notice Get a stored record by index
    function getEmail(uint256 index) public view returns (
        string memory, string memory, string memory, string memory, uint256
    ) {
        require(index < emails.length, "Index out of bounds");
        EmailRecord storage r = emails[index];
        return (r.emailHash, r.sender, r.subject, r.prediction, r.timestamp);
    }

    /// @notice Total number of stored emails
    function getEmailCount() public view returns (uint256) {
        return emails.length;
    }
}
"""

with open(SOL_FILE, "w") as f:
    f.write(solidity_code)

print(f"✅ Solidity contract written → {SOL_FILE}")

## ⚙️ Cell 9 — Compile Smart Contract

In [ ]:
import json
from solcx import compile_standard, install_solc

print("📦 Installing Solidity 0.8.0 compiler...")
install_solc("0.8.0")

with open(SOL_FILE, "r") as f:
    src = f.read()

compiled_sol = compile_standard(
    {
        "language": "Solidity",
        "sources": {"EmailRegistry.sol": {"content": src}},
        "settings": {
            "outputSelection": {
                "*": {"*": ["abi", "metadata",
                             "evm.bytecode", "evm.sourceMap"]}
            }
        },
    },
    solc_version="0.8.0",
)

with open(COMPILED_JSON, "w") as f:
    json.dump(compiled_sol, f)

BYTECODE = compiled_sol["contracts"]["EmailRegistry.sol"]["EmailRegistry"]["evm"]["bytecode"]["object"]
ABI      = compiled_sol["contracts"]["EmailRegistry.sol"]["EmailRegistry"]["abi"]

print("✅ Contract compiled. ABI & bytecode extracted.")

## 🚀 Cell 10 — Deploy Contract to Ganache

In [ ]:
from web3 import Web3

w3 = Web3(Web3.HTTPProvider(GANACHE_URL))
if not w3.is_connected():
    raise ConnectionError("❌ Cannot connect to Ganache. Is it running on port 7545?")
print("✅ Connected to Ganache")

EmailRegistry = w3.eth.contract(abi=ABI, bytecode=BYTECODE)
nonce = w3.eth.get_transaction_count(ACCOUNT_ADDRESS)

txn = EmailRegistry.constructor().build_transaction({
    "from": ACCOUNT_ADDRESS,
    "nonce": nonce,
    "gas": 3_000_000,
    "gasPrice": w3.to_wei("20", "gwei")
})

signed_txn = w3.eth.account.sign_transaction(txn, PRIVATE_KEY)
raw_txn = getattr(signed_txn, "rawTransaction",
          getattr(signed_txn, "raw_transaction", None))

tx_hash = w3.eth.send_raw_transaction(raw_txn)
receipt = w3.eth.wait_for_transaction_receipt(tx_hash)

CONTRACT_ADDRESS = receipt.contractAddress
print(f"✅ Contract deployed at: {CONTRACT_ADDRESS}")

# Persist ABI + address for later use
with open(DEPLOYMENT_JSON, "w") as f:
    json.dump({"address": CONTRACT_ADDRESS, "abi": ABI}, f, indent=2)

print(f"💾 Deployment info saved → {DEPLOYMENT_JSON}")

## 🔍 Cell 11 — Classify & Store Emails

> **Only SAFE emails are stored on the blockchain** as per the project requirement.
> Phishing emails are flagged but NOT stored.

In [ ]:
import hashlib
from email import policy
from email.parser import BytesParser

# ── Load deployment info (if re-running after a restart) ─────────────────────
with open(DEPLOYMENT_JSON, "r") as f:
    dep = json.load(f)

CONTRACT_ADDRESS = dep["address"]
ABI              = dep["abi"]
contract         = w3.eth.contract(address=CONTRACT_ADDRESS, abi=ABI)

# ── Load the best model & vectorizer ─────────────────────────────────────────
best_info = open(ARTIFACT_DIR / "best_model.txt").read()
print("Best model info:\n", best_info)

is_lstm = "best_model.keras" in best_info

with open(ARTIFACT_DIR / "tfidf_vectorizer.pkl", "rb") as f:
    vec = pickle.load(f)

if is_lstm:
    clf = tf.keras.models.load_model(ARTIFACT_DIR / "best_model.keras")
    from tensorflow.keras.preprocessing.text import tokenizer_from_json
    with open(ARTIFACT_DIR / "lstm_tokenizer.json") as f:
        clf_tok = tokenizer_from_json(f.read())
else:
    with open(ARTIFACT_DIR / "best_model.pkl", "rb") as f:
        clf = pickle.load(f)

with open(ARTIFACT_DIR / "label_mapping.json") as f:
    lm = json.load(f)
int_to_label = {int(k): v for k, v in lm["int_to_label"].items()}

# ── Prediction helper ─────────────────────────────────────────────────────────
def predict_label(text: str) -> str:
    processed = preprocess_text(text)
    if is_lstm:
        seq = pad_sequences(
            clf_tok.texts_to_sequences([processed]),
            maxlen=LSTM_MAX_LEN, padding="post", truncating="post"
        )
        pred_idx = int(np.argmax(clf.predict(seq), axis=1)[0])
    else:
        pred_idx = int(clf.predict(vec.transform([processed]))[0])
    return int_to_label[pred_idx]  # "safe" or "phishing"

# ── Email parser ──────────────────────────────────────────────────────────────
def parse_eml(filepath: str) -> dict:
    with open(filepath, "rb") as f:
        msg = BytesParser(policy=policy.default).parse(f)
    sender  = str(msg["From"]    or "unknown")
    subject = str(msg["Subject"] or "No Subject")
    if msg.is_multipart():
        body = ""
        for part in msg.walk():
            if part.get_content_type() == "text/plain":
                body = part.get_payload(decode=True).decode(errors="ignore")
                break
    else:
        body = msg.get_payload(decode=True).decode(errors="ignore")
    return {"sender": sender, "subject": subject, "body": body}

# ── Store on blockchain ───────────────────────────────────────────────────────
def store_safe_email(email_hash: str, sender: str,
                      subject: str, prediction: str):
    """Write one safe-email record to the blockchain."""
    nonce = w3.eth.get_transaction_count(ACCOUNT_ADDRESS)
    txn = contract.functions.storeEmail(
        email_hash, sender, subject, prediction
    ).build_transaction({
        "from": ACCOUNT_ADDRESS,
        "nonce": nonce,
        "gas": 3_000_000,
        "gasPrice": w3.to_wei("20", "gwei")
    })
    signed = w3.eth.account.sign_transaction(txn, PRIVATE_KEY)
    raw    = getattr(signed, "rawTransaction",
              getattr(signed, "raw_transaction", None))
    tx_hash = w3.eth.send_raw_transaction(raw)
    w3.eth.wait_for_transaction_receipt(tx_hash)
    return tx_hash.hex()

# ── Main classify-and-store function ─────────────────────────────────────────
def classify_and_store(filepath: str):
    """Classify an .eml file. Store its hash on-chain only if SAFE."""
    data       = parse_eml(filepath)
    prediction = predict_label(data["body"])
    email_hash = hashlib.sha256(data["body"].encode("utf-8")).hexdigest()

    print(f"\n📩 {os.path.basename(filepath)}")
    print(f"   From    : {data['sender']}")
    print(f"   Subject : {data['subject']}")
    print(f"   Result  : {'✅ SAFE' if prediction == 'safe' else '⚠️  PHISHING'}")

    if prediction == "safe":
        tx = store_safe_email(email_hash, data["sender"],
                               data["subject"], prediction)
        print(f"   ⛓️  Hash stored on blockchain | TxHash: {tx}")
    else:
        print("   🚫 Phishing email — NOT stored on blockchain.")

    return prediction

print("✅ classify_and_store() is ready.")

## 📂 Cell 12 — Process All Emails in the `emails/` Folder

In [ ]:
from pathlib import Path

# Path.glob() handles filenames with spaces correctly on Windows
email_dir = Path(EMAIL_FOLDER)

eml_files = sorted(email_dir.glob("*.eml"))  # returns Path objects

if not eml_files:
    print("\u26a0\ufe0f  No .eml files found in", email_dir.resolve())
    print("    Make sure your .eml files are inside the 'emails/' folder.")
else:
    print(f"Found {len(eml_files)} email(s):")
    for p in eml_files:
        print(f"  \u2022 {p.name}")
    print()

    for path in eml_files:
        if not path.exists():
            print(f"\u26a0\ufe0f  Skipping (not found): {path}")
            continue
        classify_and_store(str(path))  # pass as str for open() compatibility

## 🔎 Cell 13 — Verify Stored Emails on Blockchain

In [ ]:
def verify_email(email_body: str):
    """Check whether an email's hash is stored on the blockchain."""
    query_hash  = hashlib.sha256(email_body.encode("utf-8")).hexdigest()
    total       = contract.functions.getEmailCount().call()

    if total == 0:
        print("No emails stored yet.")
        return

    for i in range(total):
        h, sender, subject, pred, ts = contract.functions.getEmail(i).call()
        if h == query_hash:
            from datetime import datetime
            print(f"✅ Found on blockchain (record #{i})")
            print(f"   Sender    : {sender}")
            print(f"   Subject   : {subject}")
            print(f"   Prediction: {pred}")
            print(f"   Timestamp : {datetime.utcfromtimestamp(ts)} UTC")
            return

    print("❌ Email hash not found on blockchain.")

# ── Example: verify a specific .eml file ─────────────────────────────────────
# Change the filename to one of your processed safe emails
sample_path = os.path.join(EMAIL_FOLDER, "safe_email2.eml")
if os.path.exists(sample_path):
    data = parse_eml(sample_path)
    verify_email(data["body"])
else:
    print(f"Sample file not found: {sample_path}")
    print("Total emails on blockchain:", contract.functions.getEmailCount().call())